In [ ]:
# Script to pull AEO 2026 data using the API

import requests
import json
import pandas as pd
from time import sleep
import os

# Set AEO_year
AEO_year = 2026

# Get EIA API key
api_key = os.getenv('EIA_API_KEY')

In [ ]:
# In this cell, we generate a dictionary where the keys are the names of the EIA series that we want to scrape,
# and the values are lists of the corresponding seriesIds. Each series has many seriesIds associated with it.
# We will later use this data structure to construct a single API call for each series.

# List of desired series names
series_names = [
#     'Electricity Capital Costs : Nuclear : Light Water Reactor',
#     'Electricity Capital Costs : Nuclear : Small Modular Reactor',
#     'Electricity Capital Costs : Renewables : Wood and Other Biomass',
#     'Electricity Capital Costs : Coal : USC with 30% Sequestration',
#     'Electricity Capital Costs : Coal : USC with 90% Sequestration',
#     'Electricity Capital Costs : Coal : Ultra-supercritical (USC)',
#     'Electricity Capital Costs : Combined Cycle : CC with 90% Sequestration',
#     'Electricity Capital Costs : Combined Cycle : Multi shaft',
#     'Electricity Capital Costs : Combined Cycle : Single shaft',
#     'Electricity Capital Costs : Combustion Turbine : Aeroderivative',
#     'Electricity Capital Costs : Combustion Turbine : Industrial Frame',
#     'Electricity Capital Costs : Combustion Turbine : Internal Combustion',
#     'Electricity Capital Costs : Fuel Cells',
#     'Energy Prices : Nominal : Electric Power : Steam Coal',
    'Energy Prices : Electric Power : Steam Coal',
    'Energy Prices : Electric Power : Natural Gas',
#     'Energy Prices : Nominal : Electric Power : Natural Gas',
    'Energy Prices : Electric Power : Nuclear Fuel',
#     'Energy Prices : Nominal : Electric Power : Uranium',
    'Energy Use : Electric Power : Natural Gas',
    'Energy Use : Total : Natural Gas',
    'Energy Use : Delivered : All Sectors : Electricity'
 ]

# This API call will get us a list of dictionaries that associate each seriesId to a series name.
path = f'https://api.eia.gov/v2/aeo/{AEO_year}/facet/seriesId?api_key={api_key}'
resp = requests.get(path).json()
series_maps = resp['response']['facets']

# Next we construct the series_lists dictionary, filtered to the desired series in series_names.
seriesId_lists = {}
for series_map in series_maps:
    name = series_map['name']
    if name in series_names:
        id = series_map['id']
        if name not in seriesId_lists.keys():
            seriesId_lists[name] = [id]
        else:
            seriesId_lists[name] += [id]

In [ ]:
# In this cell, we obtain the regionIds for our desired regions
region_names = [
    'New England',
    'Middle Atlantic',
    'East North Central',
    'West North Central',
    'South Atlantic',
    'East South Central',
    'West South Central',
    'Mountain',
    'Pacific',
    'United States'
 ]
path = f'https://api.eia.gov/v2/aeo/{AEO_year}/facet/regionId?api_key={api_key}'
resp = requests.get(path).json()
region_maps = resp['response']['facets']
regionIds = {item['id'] for item in region_maps if str(item['name']) in region_names}
#n = [(item['id'], item['name']) for item in x if item['name'] in region_names]
assert len(regionIds) >= len(region_names)
regionIds

In [ ]:
# Inspect available scenario IDs before building the scenario filter in Cell 4
path = f'https://api.eia.gov/v2/aeo/{AEO_year}/facet/scenario?api_key={api_key}'
resp = requests.get(path).json()
scenario_maps = resp['response']['facets']
scenario_df = pd.DataFrame(scenario_maps)[['id', 'name']]
scenario_df = scenario_df.sort_values('id').reset_index(drop=True)
scenario_df

In [ ]:
# Inspect available scenario IDs before building the scenario filter in Cell 4
path = f'https://api.eia.gov/v2/aeo/{AEO_year}/facet/scenario?api_key={api_key}'
resp = requests.get(path).json()
scenario_maps = resp['response']['facets']
scenario_df = pd.DataFrame(scenario_maps)[['id', 'name']]
scenario_df = scenario_df.sort_values('id').reset_index(drop=True)
scenario_df

In [ ]:
# Scenario filter
# Use dynamic IDs for high/low economic growth so this updates with AEO_year
scenarios = [f'cb{AEO_year}', 'highogs', 'lowogs', f'hm{AEO_year}', f'lm{AEO_year}', 'altelec']

# Validate scenario IDs so naming changes are caught early
path = f'https://api.eia.gov/v2/aeo/{AEO_year}/facet/scenario?api_key={api_key}'
resp = requests.get(path).json()
available_scenarios = {item['id'] for item in resp['response']['facets']}
missing_scenarios = [s for s in scenarios if s not in available_scenarios]
if missing_scenarios:
    print('Scenario IDs not found for this AEO year:', missing_scenarios)
    print('Run the scenario lookup cell above and update scenarios.')
    raise ValueError('Invalid scenario IDs in scenarios list')
f1 = '&'.join(['facets[scenario][]=' + scenario for scenario in scenarios])

# RegionId filter
f2 = '&'.join(['facets[regionId][]=' + regionId for regionId in regionIds])

# We can't get all of the data at once because EIA limits you to 5000 rows per API call. We will
# loop over the seriesId_lists to compose a single query for each series.
df = pd.DataFrame()
for series, seriesId_list in seriesId_lists.items():
    # Filter to get all seriesIds associated with this series
    f3 = '&'.join(['facets[seriesId][]=' + seriesId for seriesId in seriesId_list])

    # Construct API call
    path = f'https://api.eia.gov/v2/aeo/{AEO_year}/data?api_key={api_key}&data[]=value&{f1}&{f2}&{f3}'
    resp = requests.get(path).json()

    # The API throttling is very aggressive. We have to slow things way down or our api_key will
    # be temporarily locked. This makes it important for us to do as few api calls as possible, or
    # else this code will require a very long runtime.
    sleep(4)

    # Verify that the call worked. If we have been throttled, then the key 'response' will be missing from the response
    if 'response' not in resp.keys():
        raise Exception(resp)
    
    # Create dataframe
    new_df = pd.DataFrame(resp['response']['data'])
    print(len(new_df))

    # Check length of dataframe.
    if len(new_df) == 5000:
        raise Exception('An API Call returned 5000 rows, which is the maximum allowed. This means there was too much data to return in a single call. More filtering is needed to reduce the amount of data per call.')
    
    # Concatenate with df
    df = pd.concat([df, new_df])

# Additional processing
df = df.drop_duplicates()
df = df.drop(columns=['history'])
df = df.rename(columns={'period': 'year', 'unit': 'units'})

In [ ]:
#Copying the dataframe to process data further
df2 = df
#df2

df2 = df2.drop(columns=['tableId', 'tableName', 'seriesId', 'regionId'])
#df2


#Scenario list
scenario_filter_orig = df2['scenario'].unique().tolist()
scenario_filter = [i.upper() for i in scenario_filter_orig]
scenario_filter

In [ ]:
#Display series list
series_names

In [ ]:
#Dictionary to generate suffixes for the output files. Original series names have spaces which make it unusable directly

file_name_suffix_list = {
    'Energy Prices : Electric Power : Steam Coal': 'coal_prices',
    'Energy Prices : Electric Power : Natural Gas': 'ng_prices',
    'Energy Prices : Electric Power : Nuclear Fuel': 'uranium_prices',
    'Energy Use : Electric Power : Natural Gas': 'ng_demand_electricity',
    'Energy Use : Total : Natural Gas': 'ng_tot_demand',
    'Energy Use : Delivered : All Sectors : Electricity': 'electricity_consumption'
}


In [ ]:
#Creating a new column concatenating multiple columns and as per standard file columns used in previous years
df2.insert(7, 'Col_header', True)
#df2

df2['Col_header'] = df2['seriesName'] + ", " + df2['regionName'] + f" AEO{AEO_year} " + df2['scenarioDescription'] + f", AEO{AEO_year}"
#df2

df2['scenario'] = df2['scenario'].str.upper()
df2

In [ ]:
#Loop to filter through each series name and scenario and write filtered dataframe to a separate csv file
#Within loop pivot the dataframe to re-align such that Prices from each region become columns


#df_new = {}

# Specify outputs location
output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)

for x in series_names:
    for j in scenario_filter:
        df_filter = pd.DataFrame(df2.loc[(df2['seriesName'] == x) & (df2['scenario'] == j)])
        df_pivot_filtered = df_filter.pivot(index='year', columns='Col_header', values='value')
        df_pivot_filtered.reset_index(inplace=True)
        if 'Prices' in x:
            df_pivot_filtered['units'] = f'{AEO_year-1} $/MMBtu'
        else:
            df_pivot_filtered['units'] = 'quads'

        # Find the column with 'United States' in its name
        us_col = next(col for col in df_pivot_filtered.columns if "United States" in col)

        # Remove it from its current position
        cols = [col for col in df_pivot_filtered.columns if col != us_col]

        # Insert it as the second to last column (i.e., right before the units column)
        cols.insert(-1, us_col)

        # Reorder the DataFrame
        df_pivot_filtered = df_pivot_filtered[cols]
        
        fileseries_suffix = file_name_suffix_list.get(x, 'Unknown')
        
        # Output files
        filename = os.path.join(output_dir, f"AEO_{j}_{AEO_year}_{fileseries_suffix}.csv")
        
        # Only output coal and uranium prices for REF scenario
        # if x == 'Energy Prices : Electric Power : Steam Coal' or x == 'Energy Prices : Electric Power : Uranium':
        #     if j == f'REF{AEO_year}':
        #         df_pivot_filtered.to_csv(filename, index=False)
                
        # else:
        #     df_pivot_filtered.to_csv(filename, index=False)
        df_pivot_filtered.to_csv(filename, index=False)
            

print('Scrape complete! Files generated!')